# Graph Attention Network (GAT)

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

GAT substitui a normalização fixa do GCN por *atenção*: cada aresta ganha um peso aprendido que depende das features de origem e destino. Assim o modelo decide quais vizinhos importam.


## Formulação Matemática

$$e_{ij} = \text{LeakyReLU}(\mathbf{a}^\top [Wh_i \,\Vert\, Wh_j])$$

$$\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k \in \mathcal{N}(i)} \exp(e_{ik})}$$

$$h'_i = \sigma\!\left(\sum_{j \in \mathcal{N}(i)} \alpha_{ij}\,W h_j\right)$$


## Implementação


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
class GATLayer(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        self.W = nn.Linear(in_f, out_f, bias=False)
        self.a = nn.Linear(2 * out_f, 1, bias=False)
    def forward(self, X, A):
        N = X.size(0)
        Wh = self.W(X)                            # (N, out)
        Wh_i = Wh.unsqueeze(1).expand(-1, N, -1)  # (N, N, out)
        Wh_j = Wh.unsqueeze(0).expand(N, -1, -1)
        pair = torch.cat([Wh_i, Wh_j], dim=-1)    # (N, N, 2*out)
        e = F.leaky_relu(self.a(pair).squeeze(-1), negative_slope=0.2)
        e = e.masked_fill(A == 0, float('-inf'))
        alpha = F.softmax(e, dim=-1)
        return alpha @ Wh, alpha


## Experimento


In [ ]:
torch.manual_seed(0)
X = torch.randn(6, 4)
A = torch.tensor([
    [1,1,1,0,0,0],
    [1,1,1,0,0,0],
    [1,1,1,1,0,0],
    [0,0,1,1,1,1],
    [0,0,0,1,1,1],
    [0,0,0,1,1,1],
], dtype=torch.float)
gat = GATLayer(in_f=4, out_f=8)
H, alpha = gat(X, A)
print('output:', H.shape)
print('attention row sums:', alpha.sum(-1))


## Discussão

- GAT multi-head média ou concatena várias cabeças, igual a Transformers.
- Para grafos grandes use implementações *esparsas* (atenção indexada por aresta) — o tensor denso `(N, N)` aqui é só para clareza.
- GAT funciona especialmente bem quando a importância dos vizinhos é heterogênea (e.g. grafos sociais).


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
